# 🧠 03 - Baseline CNN

## Sıfırdan Eğitilmiş Basit Konvolüsyonel Sinir Ağı

Transfer learning kullanmadan sıfırdan basit bir CNN modeli eğitir. 
Kıyaslama (baseline) sağlar.

### Sonuç
- Test Accuracy: **%85.76**
- F1-Score: **0.8571**
- Model Boyutu: **39.61 MB**
- Parametre: **3.45M**

In [ ]:
# ============================================================
# 1. HAZIRLIK
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 10
drive_proje = "/content/drive/MyDrive/Domates_Projesi"

if not os.path.exists("tomato_data"):
    shutil.copytree(f"{drive_proje}/data", "tomato_data")

print(f"GPU sayısı: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# ============================================================
# 2. DATAGENERATOR
# ============================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=20,
    width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, zoom_range=0.1, fill_mode='nearest'
)
valid_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    "tomato_data/train", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True, seed=42
)
valid_generator = valid_datagen.flow_from_directory(
    "tomato_data/valid", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    "tomato_data/test", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print("✅ Generator'lar hazır")

In [ ]:
# ============================================================
# 3. BASELINE CNN MİMARİSİ
# ============================================================

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

baseline_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
], name='Baseline_CNN')

baseline_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Toplam parametre: {baseline_model.count_params():,}")
baseline_model.summary()

In [ ]:
# ============================================================
# 4. EĞİTİM (15 epoch)
# ============================================================

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

os.makedirs("models", exist_ok=True)

callbacks = [
    ModelCheckpoint('models/baseline_cnn.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
]

baseline_history = baseline_model.fit(
    train_generator,
    epochs=15,
    validation_data=valid_generator,
    callbacks=callbacks,
    verbose=1
)

print("✅ Eğitim tamamlandı")

In [ ]:
# ============================================================
# 5. TEST DEĞERLENDİRMESİ
# ============================================================

from tensorflow.keras.models import load_model
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

baseline_best = load_model('models/baseline_cnn.keras')

test_generator.reset()
test_loss, test_accuracy = baseline_best.evaluate(test_generator, verbose=1)

test_generator.reset()
predictions = baseline_best.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
model_size = os.path.getsize('models/baseline_cnn.keras') / (1024 * 1024)

print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"F1-Score:      {f1:.4f}")
print(f"Boyut:         {model_size:.2f} MB")

print("\n🔬 KIYASLAMA:")
print(f"   Baseline CNN:   {test_accuracy*100:.2f}%")
print(f"   MobileNetV2:    %92.40")
print(f"   EfficientNetB0: %96.55")
print(f"   ResNet50:       %98.65")

In [ ]:
# ============================================================
# 6. DRIVE'A YEDEKLE
# ============================================================

shutil.copy("models/baseline_cnn.keras", f"{drive_proje}/models/baseline_cnn.keras")
print("✅ Drive'a yedeklendi")

## ✅ Tamamlandı

### Karşılaştırma

| Yaklaşım | Test Acc | F1-Score |
|----------|----------|----------|
| **Baseline CNN** | %85.76 | 0.8571 |
| MobileNetV2 | %92.40 | 0.9242 |
| EfficientNetB0 | %96.55 | 0.9657 |
| ResNet50 | %98.65 | 0.9865 |

Transfer learning katkısı: **+%12.89 puan**